# LLM Web Search

Answer queries from an LLM using data that it was not originally trained on.

LMs have a **knowledge cutoff** — they know nothing about events after their training data was collected. This notebook builds a minimal "search-augmented generation" pipeline that grounds the model's answer in fresh web content.

**Pipeline:**
1. **Search** - Send the query to the [Serper](https://serper.dev) Google Search API and collect the top result links.
2. **Scrape** - Download each page and extract readable text with `BeautifulSoup`.
3. **Generate** - Pass the scraped articles plus the question to an LLM, instructing it to answer only from the provided articles.

**Requirements:**
- `requests`, `beautifulsoup4`, `tqdm`, `google-genai`
- API keys stored as Colab secrets: `SERPER_API_KEY` and `GEMINI_API_KEY`

###Get the latest links for the query using *Serper*

In [10]:
import requests
import json
from google.colab import userdata
SERPER_API_KEY = userdata.get('SERPER_API_KEY')

query = "What is the latest new on Anthropic IPO?"

url = "https://google.serper.dev/search"
payload = json.dumps({ "q": query, "num": 5 })

headers = {
    "X-API-KEY": SERPER_API_KEY,
    "Content-Type": "application/json"
    }

response = requests.post(url, headers=headers, data=payload)
data = response.json()

data

{'searchParameters': {'q': 'What is the latest new on Anthropic IPO?',
  'type': 'search',
  'num': 5,
  'engine': 'google'},
 'organic': [{'title': "Anthropic IPO live updates: AI lab's S-1 filing",
   'link': 'https://www.cnbc.com/2026/10/26/anthropic-ipo-live-updates-ai-s-1-filing.html',
   'snippet': "Anthropic's IPO plans: 900% revenue retention, but $80 billion in first-half losses. Anthropic reports $80.7 billion loss in first half of 2026 ...",
   'date': '1 day ago',
   'position': 1},
  {'title': 'Anthropic IPO live updates: S-1 filing discloses $16.3B in ...',
   'link': 'https://finance.yahoo.com/technology/live/anthropic-ipo-live-updates-frontier-ai-lab-discloses-revenue-and-risks-in-highly-anticipated-s-1-filing-141702607.html',
   'snippet': "S-1 filing discloses $16.3B in first-half revenue, plans to list as 'ANT' was valued at $965 billion, Anthropic (ANTH.PVT) made $974 million ...",
   'date': '1 day ago',
   'position': 2},
  {'title': 'Anthropic IPO Timeline and Fi

In [11]:
data.keys()

dict_keys(['searchParameters', 'organic', 'peopleAlsoAsk', 'relatedSearches', 'credits'])

In [12]:
links = [link['link'] for link in data["organic"]]
links

['https://www.cnbc.com/2026/10/26/anthropic-ipo-live-updates-ai-s-1-filing.html',
 'https://finance.yahoo.com/technology/live/anthropic-ipo-live-updates-frontier-ai-lab-discloses-revenue-and-risks-in-highly-anticipated-s-1-filing-141702607.html',
 'https://forgeglobal.com/anthropic_ipo/',
 'https://apnews.com/article/anthropic-ipo-shares-public-ca2f4c282bef3201a392e59e3df2300c',
 'https://www.nbcnews.com/business/markets/anthropic-releases-ipo-filing-ahead-highly-anticipated-stock-debut-rcna605676']

###Fetch the data from links using BeautifulSoup

In [34]:
from bs4 import BeautifulSoup
from tqdm import tqdm

# Define standard headers to look like a real browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
}

scrapped_data = []
scrapped_link = []

for link in tqdm(links):
  response = requests.get(link, headers=headers)

  if response.status_code != 200:
    print(f"Failed to fetch {link}. Status code: {response.status_code}")
    continue

  soup = BeautifulSoup(response.text, "html.parser")

  # Remove script/style tags
  for tag in soup(["script", "style"]):
      tag.extract()

  # Get all text
  text = soup.get_text(separator=" ", strip=True)
  scrapped_data.append(text)
  scrapped_link.append(link)


 60%|██████    | 3/5 [00:02<00:01,  1.36it/s]

Failed to fetch https://apnews.com/article/anthropic-ipo-shares-public-ca2f4c282bef3201a392e59e3df2300c. Status code: 403


100%|██████████| 5/5 [00:02<00:00,  1.99it/s]


In [38]:
len(scrapped_data)

4

In [39]:
for d in scrapped_data:
  print(d[:100])
  print()

Anthropic IPO live updates: AI lab's S-1 filing Skip Navigation Markets Pre-Markets U.S. Markets Cur

Anthropic IPO live updates: S-1 filing discloses $16.3B in first-half revenue, plans to list as 'ANT

Anthropic IPO Timeline and Financing Details - Forge PRIVATE MARKET UPDATE The $20 billion threshold

Anthropic IPO filing reveals $80 billion net loss, spending spree IE 11 is not supported. For an opt



###Get response to query from LLM using scrapped data

In [42]:
from google import genai
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

prompt = '''Your task is to take articles from multiple news channels that attached
and answer the question.
Do answer in less than 500 words.
ALWAYS provide citations.
Do NOT add your owm commentary.
Do NOT add information outside of the articles.
''' + '\n\n\nQuestion: ' + query + '\n\n\nArticles: '

for i in range(len(scrapped_link)):
  prompt += scrapped_link[i] + '\n\n\n'
  prompt += scrapped_data[i] + '\n\n\n'



client = genai.Client(api_key=GEMINI_API_KEY)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

In [44]:
print(response.text)

Anthropic officially filed its Form S-1 prospectus with the Securities and Exchange Commission (SEC) on Monday, October 26, 2026, signaling its intent to go public on the Nasdaq under the ticker symbol "ANT" (CNBC, "Anthropic IPO live updates: AI lab's S-1 filing," Mon, Oct 26 2026 8:27 PM EDT; Yahoo Finance, "Anthropic IPO live updates: S-1 filing discloses $16.3B in first-half revenue, plans to list as 'ANT'," Mon, October 26, 2026 at 2:47 PM MDT). This follows a confidential filing in June 2026 (CNBC, "Anthropic IPO live updates: AI lab's S-1 filing," Mon, Oct 26 2026 8:27 PM EDT). The offering is widely anticipated to occur in mid-November 2026, with Goldman Sachs and Morgan Stanley leading as the top underwriters (NBC News, "Anthropic IPO filing reveals $80 billion net loss, spending spree," Oct. 26, 2026, 9:12 AM EDT; CNBC, "Anthropic IPO live updates: AI lab's S-1 filing," Mon, Oct 26 2026 9:10 AM EDT).

For the first half of 2026, Anthropic reported $16.29 billion in revenue, a

###Conclusions

- **Fresh knowledge without retraining:** Combining a search API with an LLM lets the model answer questions about recent events (e.g. the Anthropic IPO) that fall after its training cutoff.
- **Grounding reduces hallucination:** Restricting the prompt to "answer only from the attached articles, no outside information" keeps the response tied to verifiable sources.
- **Simple but effective:** The whole pipeline is search → scrape → prompt, with no vector database or embeddings — the large context window of Gemini 2.5 Flash can absorb several full articles directly.

**Limitations & next steps:**
- Raw page text includes navigation, ads and boilerplate; better extraction (e.g. `trafilatura`, `readability`) would yield cleaner context.
- Some sites block scrapers or render content with JavaScript, so certain links may return little useful text.
- Large pages can bloat the prompt — chunking + embedding retrieval (RAG) or truncation would scale better.
- Returning source URLs alongside the answer would add citations and improve trust.
- Adding error handling/timeouts for failed requests would make the pipeline more robust.